In [ ]:
import numpy as np
import pyref.fitting as fit
from refnx.analysis import Parameter, Parameters

from utils import read_fit, read_ooc, read_xrr
from utils.models import AnisotropyObjective, CurveFitter, ReflectModel
from utils.profile_slab import (
    AdaptiveBookendedOrientationDensityProfile,
    AdaptiveOrientationDensityProfile,
    AdaptiveOrientationProfile,
)


In [ ]:
oocs = read_ooc("dft.csv", material="znpc")
data = read_xrr("reflectivity_data", material="znpc", source="hub")
dft_fit = read_fit("dft/dft_en_offset_new2.pkl", material="znpc", source="local")

In [ ]:
free_fit = read_fit("free/free_en_offset_init_2.pkl", material="znpc", source="local")

In [ ]:
print(free_fit.objectives[0].model.structure)

In [ ]:
energies = [o.model.energy for o in dft_fit.objectives]
mo = [i for i, o in enumerate(dft_fit.objectives) if o.model.energy == 283.7][0]
template = dft_fit.objectives[mo]
template.plot()

In [ ]:
def en_access(energy, dft_fit):
    try:
        return next(o for o in dft_fit.objectives if o.model.energy == energy)
    except StopIteration:
        raise ValueError(f"No objective found for energy {energy}")

def model_bookended(
    energy: float | int | str,
    characteristic_thickness: float = .4,
    num_slabs: int = 20,
):
    energy = float(energy)
    template = en_access(energy, dft_fit)
    surf = template.model.structure.components[1]
    bulk = template.model.structure.components[2]
    inter = template.model.structure.components[3]
    total_thick = bulk.thick.value + inter.thick.value + surf.thick.value
    surface_roughness = surf.rough.value
    bulk_density = bulk.sld.density.value
    si_density = inter.sld.density.value
    vac_density = surf.sld.density.value
    bulk_angle = bulk.sld.rotation.value
    si_angle = inter.sld.rotation.value
    vac_angle = surf.sld.rotation.value
    energy_offset = template.model.energy_offset.value

    return (
        template.model.structure.components[0]
        | AdaptiveBookendedOrientationDensityProfile(
            oocs,
            energy=energy,
            total_thick=total_thick,
            surface_roughness=surface_roughness,
            density_bulk=bulk_density,
            density_si=si_density,
            density_vac=vac_density,
            tau_si=inter.thick.value,
            tau_vac=surf.thick.value,
            alpha_bulk=bulk_angle,
            alpha_si=si_angle,
            alpha_vac=vac_angle,
            num_slabs=num_slabs,
            energy_offset=energy_offset,
            name=f"ZnPc_{energy:.1f}",
        )
        | template.model.structure.components[4]
        | template.model.structure.components[5]
    )




def model_orientation(
    energy: float | int | str,
    characteristic_thickness: float = .4,
    num_slabs: int = 20,
):
    energy = float(energy)
    template = en_access(energy, dft_fit)
    surf = template.model.structure.components[1]
    bulk = template.model.structure.components[2]
    inter = template.model.structure.components[3]
    total_thick = bulk.thick.value + inter.thick.value
    surface_roughness = bulk.rough.value
    initial_angle = inter.sld.rotation.value
    max_density = bulk.sld.density.value
    max_angle = bulk.sld.rotation.value
    energy_offset = template.model.energy_offset.value

    return (
        template.model.structure.components[0]
        | surf
        | AdaptiveOrientationProfile(
            oocs,
            energy=energy,
            energy_offset=energy_offset,
            density=max_density,
            total_thick=total_thick,
            surface_roughness=surface_roughness,
            characteristic_thickness=characteristic_thickness,
            max_angle=max_angle,
            initial_angle=initial_angle,
            num_slabs=num_slabs,
        )
        | template.model.structure.components[4]
        | template.model.structure.components[5]
    )

def model_density(
    energy: float | int | str,
    characteristic_thickness: float = .4,
    num_slabs: int = 20,
):
    energy = float(energy)
    template = en_access(energy, dft_fit)
    surf = template.model.structure.components[1]
    bulk = template.model.structure.components[2]
    inter = template.model.structure.components[3]
    total_thick = bulk.thick.value + inter.thick.value
    surface_roughness = bulk.rough.value
    max_density = bulk.sld.density.value
    initial_density = inter.sld.density.value
    initial_angle = inter.sld.rotation.value
    max_angle = bulk.sld.rotation.value - initial_angle
    energy_offset = template.model.energy_offset.value

    return (
        template.model.structure.components[0]
        | template.model.structure.components[1]
        | AdaptiveOrientationDensityProfile(
            oocs,
            energy=energy,
            energy_offset=energy_offset,
            total_thick=total_thick,
            surface_roughness=surface_roughness,
            characteristic_thickness=characteristic_thickness,
            max_angle=max_angle,
            initial_angle=initial_angle,
            max_density=max_density,
            initial_density=initial_density,
            num_slabs=num_slabs,
        )
        | template.model.structure.components[4]
        | template.model.structure.components[5]
    )

def model_dft(
    energy: float | int | str,
):
    energy = float(energy)
    template = en_access(energy, dft_fit)
    return (
        template.model.structure.components[0]
        | template.model.structure.components[1]
        | template.model.structure.components[2]
        | template.model.structure.components[3]
        | template.model.structure.components[4]
        | template.model.structure.components[5]
    )

models = {e: model_orientation(e) for e in data}


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(nrows=5, sharex=True, sharey=True, figsize=(5, 15))

model_dft(283.7).plot(ax=ax[0])
ax[0].set_title("Rebuilt from DFT Fit Results")
ax[0].set_xlabel("")
ax[0].legend(loc="upper center", ncol=2)
en_access(283.7, dft_fit).model.structure.plot(ax=ax[1])
ax[1].set_title("As read from DFT Fit Results")
ax[1].set_xlabel("")
# Remove legends
ax[1].legend().remove()
model_orientation(283.7).plot(ax=ax[2])
ax[2].set_title("Orientation Profile")
ax[2].set_xlabel("")
ax[2].legend().remove()
model_density(283.7).plot(ax=ax[3])
ax[3].set_title("Density Profile")
ax[3].legend().remove()
ax[3].set_xlim(None, 210)
ax[3].set_xlabel("")
ax[3].set_ylim(-2e-3, 5.5e-3)
model_bookended(283.7, num_slabs=100).plot(ax=ax[4])
ax[4].set_title("Bookended Profile")
ax[4].set_xlabel("")
ax[4].legend().remove()
plt.show()

In [ ]:
models = {e: model_bookended(e, num_slabs=100) for e in data}

In [ ]:
from scipy.stats import loguniform, truncnorm

template = models["283.7"]
znpc = template.components[1]

# Surface Layer (bookended model has no separate surface slab)
# template.components[1] is AdaptiveBookendedOrientationDensityProfile

# ZnPc Layer
znpc.total_thick.setp(vary=True, bounds=(160, 210))
# znpc.surface_roughness.setp(vary=True, bounds=(0, 12))
znpc.tau_si.setp(vary=True, bounds=(0, 15))
znpc.tau_vac.setp(vary=True, bounds=(0, 15))
znpc.alpha_bulk.setp(vary=True, bounds=(0, np.pi/2))
znpc.alpha_si.setp(vary=True, bounds=(0, np.pi / 2))
znpc.alpha_vac.setp(vary=True, bounds=(0, np.pi / 2))
znpc.density_bulk.setp(vary=True, bounds=(1.0, 1.7))
znpc.density_si.setp(vary=True, bounds=(1.0, 1.7))
znpc.density_vac.setp(vary=True, bounds=(1.0, 1.7))
# znpc.num_slabs is not a Parameter

# SiO2 Layer
template.components[2].thick.setp(vary=False, bounds=(0, 25))
template.components[2].rough.setp(vary=False, bounds=(0, 25))
template.components[2].sld.density.setp(vary=False)


In [ ]:
import matplotlib.pyplot as plt

refl: dict[str, fit.ReflectModel] = {
    e: fit.ReflectModel(models[e], energy=float(e), pol="sp") for e in data
}
fig, ax = plt.subplots(nrows=2)
for e, ref in refl.items():
    temp = en_access(float(e), dft_fit).model
    ref.scale_s.setp(value = temp.scale_s.value, vary=True, bounds=(0.5, 2))
    ref.scale_p.setp(value = temp.scale_p.value, vary=True, bounds=(0.5, 2))
    ref.theta_offset_s.setp(value = temp.theta_offset_s.value, vary=True, bounds=(-1.5, 1.5))
    ref.theta_offset_p.setp(value = temp.theta_offset_p.value, vary=True, bounds=(-1.5, 1.5))
    ref.energy_offset.setp(value = temp.energy_offset.value, vary=False, bounds=(-.5, .5))
    if e in ["250.0", "283.7"]:
        exp = data[e].p
        ref.bkg.setp(value=exp.y.min(), vary=False)
        i = ["250.0", "283.7"].index(e)
        ax[i].plot(exp.x, exp.y)
        ax[i].set_yscale("log")
        ax[i].axhline(
            ref.bkg.value,
            color="k",
            linestyle="--",
        )
    if float(e) >= 283.7:
        print(ref.energy_offset)

In [ ]:
model_density(283.7).plot()

In [ ]:
o = fit.AnisotropyObjective(
    refl["283.7"], data["283.7"], logp_anisotropy_weight=0.5, transform=fit.Transform("logY")
)
o.plot()
plt.show()
o.model.structure.plot()
plt.show()

In [ ]:
fitter = fit.CurveFitter(o)
fitter.fit(method="differential_evolution", workers=8)
o.plot()

In [ ]:
o.model.structure.plot()

In [ ]:
print(o.model.structure)

In [ ]:
for v in o.varying_parameters():
    bounds = v.bounds
    stderr = v.stderr
    val = v.value
    err_bounds = (val - stderr, val + stderr)
    # Check for running into the bounds
    if err_bounds[0] < bounds.lb:
        print(f"\033[91m{v.name}: {val:2g}-{stderr:2g} < {bounds.lb:2g}\033[0m")

    if err_bounds[1] > bounds.ub:
        print(f"\033[91m{v.name}: {val:2g}+{stderr:2g} > {bounds.ub:2g}\033[0m")
    else:
        print(f"\033[92m{v.name}: {val:2g}+/-{stderr:2g} in {bounds}\033[0m")

In [ ]:
import dynesty

In [ ]:
import dynesty

ndim = len(o.varying_parameters())

with dynesty.pool.Pool(8, o.logl, o.prior_transform) as pool:
    nested_sampler = dynesty.DynamicNestedSampler(
        pool.loglike,
        pool.prior_transform,
        ndim=ndim,
        nlive=25 * ndim,
        pool=pool,
        bound="multi",
        sample="rslice",
        slices=5,
        bootstrap=0,                                   # was 20, irrelevant for slice
        enlarge=1.25,                                  # fixed cushion replaces bootstrap
        first_update={"min_ncall": 2 * 25 * ndim, "min_eff": 50.0},
    )
    nested_sampler.run_nested(
        nlive_init=25 * ndim,
        nlive_batch=10 * ndim,
        maxbatch=10,
        wt_kwargs={"pfrac": 0.8},
        dlogz_init=0.5,                                # relax from default 0.01
        print_progress=True,
    )

results = nested_sampler.results

In [ ]:
from dynesty import plotting as dyplot

param_names = [p.name for p in o.varying_parameters()]

dyplot.runplot(results)
dyplot.traceplot(results, labels=param_names)
dyplot.cornerplot(results, labels=param_names)
print(results.summary())

In [ ]:
import numpy as np
from dynesty import plotting as dyplot

param_names = [p.name for p in o.varying_parameters()]

dyplot.runplot(results)
dyplot.traceplot(results, labels=param_names)
dyplot.cornerplot(results, labels=param_names)
print(results.summary())


def x0_from_dynesty_results(results, criterion: str = "logl") -> np.ndarray:
    """Build initial x0 from the most probable dynesty sample.

    Parameters
    ----------
    results
        ``nested_sampler.results`` from dynesty.
    criterion : {"logl", "log_posterior"}, optional
        ``logl`` selects the maximum-likelihood sample.
        ``log_posterior`` selects the sample with the largest
        ``logl + logwt`` (unnormalized log posterior weight).

    Returns
    -------
    numpy.ndarray
        Parameter vector ordered like ``o.varying_parameters()``.
    """
    if criterion == "logl":
        score = results.logl
    elif criterion == "log_posterior":
        score = results.logl + results.logwt
    else:
        msg = "criterion must be 'logl' or 'log_posterior'"
        raise ValueError(msg)

    idx = int(np.argmax(score))
    return np.asarray(results.samples[idx], dtype=float)


def apply_x0_to_objective(objective, x0: np.ndarray) -> None:
    """Set varying parameter values from an x0 vector."""
    for param, value in zip(objective.varying_parameters(), x0, strict=True):
        param.value = float(value)


x0 = x0_from_dynesty_results(results, criterion="logl")
for name, value in zip(param_names, x0, strict=True):
    print(f"{name}: {value:.6g}")

apply_x0_to_objective(o, x0)


In [ ]:
def x0_from_dynesty_map(results, objective, logz_index: int = -1) -> np.ndarray:
    """
    Pick the joint MAP sample from dynesty results and return it as x0
    in the same order as objective.varying_parameters().
    """
    varying = objective.varying_parameters()

    # Posterior weights, consistent with how you already compute them in the notebook
    weights = np.exp(results.logwt - results.logz[logz_index])

    idx = int(np.argmax(weights))
    x0 = np.asarray(results.samples[idx], dtype=float)

    if x0.shape != (len(varying),):
        raise ValueError(
            f"Unexpected x0 shape {x0.shape}; expected {(len(varying),)}"
        )
    return x0


def apply_x0_to_objective(objective, x0: np.ndarray) -> None:
    for p, v in zip(objective.varying_parameters(), x0, strict=True):
        p.value = float(v)


param_names = [p.name for p in o.varying_parameters()]
x0 = x0_from_dynesty_map(results, o, logz_index=-1)

for name, v in zip(param_names, x0, strict=True):
    print(f"{name}: {v:.6g}")

apply_x0_to_objective(o, x0)

fitter = fit.CurveFitter(o)
res = fitter.fit(options={"workers": 8})

In [ ]:
import numpy as np
from dynesty.utils import resample_equal
from refnx.analysis import Interval


def update_bounds_from_posterior(
    objective,
    results,
    cr_level=0.997,
    margin=0.25,
    rail_threshold=0.05,
    edge_fraction=0.02,
    expansion_factor=0.5,
    log_names=(),
    physical_floors=None,
    physical_ceilings=None,
    skip_names=(),
    verbose=True,
):
    posterior = resample_equal(
        results.samples,
        np.exp(results.logwt - results.logz[-1]),
    )

    physical_floors = physical_floors or {}
    physical_ceilings = physical_ceilings or {}
    varying = objective.varying_parameters()
    report = {}

    fmt = "{:<32} {:<10} {:<26} {:<26} {:>8} {:>8}"
    if verbose:
        print(fmt.format("parameter", "action", "old", "new", "rail_lo", "rail_hi"))
        print("-" * 116)

    for i, p in enumerate(varying):
        name = p.name
        if name in skip_names:
            continue

        s = posterior[:, i]
        use_log = name in log_names
        x = np.log(s) if use_log else s

        # extract current bounds, handling Interval, refnx PDF, and raw scipy
        b = p.bounds
        if hasattr(b, "lb") and hasattr(b, "ub"):
            lo_cur, hi_cur = float(b.lb), float(b.ub)
        elif hasattr(b, "rv"):
            sup = b.rv.support()
            lo_cur, hi_cur = float(sup[0]), float(sup[1])
        elif hasattr(b, "support"):
            sup = b.support()
            lo_cur, hi_cur = float(sup[0]), float(sup[1])
        else:
            lo_cur, hi_cur = float(np.min(s)), float(np.max(s))

        if not np.isfinite(lo_cur):
            lo_cur = float(np.min(s))
        if not np.isfinite(hi_cur):
            hi_cur = float(np.max(s))

        cl = np.log(lo_cur) if use_log else lo_cur
        ch = np.log(hi_cur) if use_log else hi_cur
        span = ch - cl

        edge_lo = float(np.mean(x < cl + edge_fraction * span))
        edge_hi = float(np.mean(x > ch - edge_fraction * span))
        rail_lo = edge_lo > rail_threshold
        rail_hi = edge_hi > rail_threshold

        alpha = (1.0 - cr_level) / 2.0
        cr_lo, cr_hi = np.quantile(x, [alpha, 1.0 - alpha])
        cr_span = cr_hi - cr_lo

        new_lo = cr_lo - margin * cr_span
        new_hi = cr_hi + margin * cr_span

        if rail_lo:
            new_lo = cl - expansion_factor * span
        if rail_hi:
            new_hi = ch + expansion_factor * span

        if use_log:
            new_lo, new_hi = float(np.exp(new_lo)), float(np.exp(new_hi))

        if name in physical_floors:
            new_lo = max(new_lo, physical_floors[name])
        if name in physical_ceilings:
            new_hi = min(new_hi, physical_ceilings[name])

        if new_hi <= new_lo:
            mid = float(np.median(s))
            half = 0.05 * max(abs(mid), 1.0)
            new_lo, new_hi = mid - half, mid + half

        p.bounds = Interval(new_lo, new_hi)
        if not (new_lo <= p.value <= new_hi):
            p.value = float(np.clip(np.median(s), new_lo, new_hi))

        action = "expand" if (rail_lo or rail_hi) else "tighten"
        report[name] = {
            "old": (lo_cur, hi_cur),
            "new": (new_lo, new_hi),
            "action": action,
            "rail_lo": rail_lo,
            "rail_hi": rail_hi,
            "edge_lo_mass": edge_lo,
            "edge_hi_mass": edge_hi,
            "median": float(np.median(s)),
            "p2.5": float(np.quantile(s, 0.025)),
            "p97.5": float(np.quantile(s, 0.975)),
        }

        if verbose:
            print(fmt.format(
                name[:32], action,
                f"[{lo_cur:.3g}, {hi_cur:.3g}]",
                f"[{new_lo:.3g}, {new_hi:.3g}]",
                "yes" if rail_lo else "",
                "yes" if rail_hi else "",
            ))

    return report

In [ ]:
report = update_bounds_from_posterior(
    o,
    results,
    log_names={
        "Surface_283.7_rough",
        "characteristic_thickness",
    },
    physical_floors={
        "Surface_283.7_density": 0.5,
        "Surface_283.7_thick": 0.0,
        "Surface_283.7_rough": 0.05,
        "Surface_283.7_rotation": 0.0,
        "total_thick": 100.0,
        "surface_roughness": 0.0,
        "characteristic_thickness": 0.1,
        "max_angle": 0.0,
        "initial_angle": 0.0,
    },
    physical_ceilings={
        "Surface_283.7_rotation": np.pi / 2,
        "max_angle": np.pi / 2,
        "initial_angle": np.pi / 4,
    },
    skip_names={"energy_offset"},
)

In [ ]:
import dynesty

ndim = len(o.varying_parameters())

with dynesty.pool.Pool(8, o.logl, o.prior_transform) as pool:
    nested_sampler = dynesty.DynamicNestedSampler(
        pool.loglike,
        pool.prior_transform,
        ndim=ndim,
        nlive=25 * ndim,
        pool=pool,
        bound="multi",
        sample="rslice",
        slices=5,
        bootstrap=0,                                   # was 20, irrelevant for slice
        enlarge=1.25,                                  # fixed cushion replaces bootstrap
        first_update={"min_ncall": 2 * 25 * ndim, "min_eff": 50.0},
    )
    nested_sampler.run_nested(
        nlive_init=25 * ndim,
        nlive_batch=10 * ndim,
        maxbatch=10,
        wt_kwargs={"pfrac": 0.8},
        dlogz_init=0.5,                                # relax from default 0.01
        print_progress=True,
    )

results = nested_sampler.results

In [ ]:
import pickle

with open("dynesty_results.pkl", "wb") as f:
    pickle.dump(results, f)

In [ ]:
import numpy as np
from dynesty import plotting as dyplot

param_names = [p.name for p in o.varying_parameters()]

dyplot.runplot(results)
dyplot.traceplot(results, labels=param_names)
dyplot.cornerplot(results, labels=param_names)
print(results.summary())


def x0_from_dynesty_results(results, criterion: str = "logl") -> np.ndarray:
    """Build initial x0 from the most probable dynesty sample.

    Parameters
    ----------
    results
        ``nested_sampler.results`` from dynesty.
    criterion : {"logl", "log_posterior"}, optional
        ``logl`` selects the maximum-likelihood sample.
        ``log_posterior`` selects the sample with the largest
        ``logl + logwt`` (unnormalized log posterior weight).

    Returns
    -------
    numpy.ndarray
        Parameter vector ordered like ``o.varying_parameters()``.
    """
    if criterion == "logl":
        score = results.logl
    elif criterion == "log_posterior":
        score = results.logl + results.logwt
    else:
        msg = "criterion must be 'logl' or 'log_posterior'"
        raise ValueError(msg)

    idx = int(np.argmax(score))
    return np.asarray(results.samples[idx], dtype=float)


def apply_x0_to_objective(objective, x0: np.ndarray) -> None:
    """Set varying parameter values from an x0 vector."""
    for param, value in zip(objective.varying_parameters(), x0, strict=True):
        param.value = float(value)


x0 = x0_from_dynesty_results(results, criterion="logl")
for name, value in zip(param_names, x0, strict=True):
    print(f"{name}: {value:.6g}")

apply_x0_to_objective(o, x0)


In [ ]:
from refnx.analysis import process_chain

logZdynesty = nested_sampler.results.logz[-1]  # value of logZ
weights = np.exp(nested_sampler.results.logwt - logZdynesty)
chain = dynesty.utils.resample_equal(nested_sampler.results.samples, weights)
process_chain(o, chain[:, None, :])

print(o.varying_parameters())

In [ ]:
objective: list[fit.AnisotropyObjective] = [
    fit.AnisotropyObjective(
        refl[e], data[e], logp_anisotropy_weight=0.5, transform=fit.Transform("logY")
    )
    for e in data if float(e) >= 283.7
]
global_objective = fit.GlobalObjective(objective)
print(global_objective.varying_parameters())

In [ ]:
from time import perf_counter

import pandas as pd
from uncertainties import ufloat

from utils.helpers.fitting_helper import aic, bic, reduced_chi2


def package_results(fitter) -> pd.DataFrame:
    params = []
    for p in fitter.objective.varying_parameters():
        err = p.stderr if p.stderr is not None else 0
        params.append(
            {
                "name": p.name,
                "value": ufloat(p.value, err),
            }
        )
    # Add fit statistics
    params.append(
        {
            "name": "reduced_chi2",
            "value": reduced_chi2(fitter.objective),
        }
    )
    params.append(
        {
            "name": "aic",
            "value": aic(fitter.objective),
        }
    )
    params.append(
        {
            "name": "bic",
            "value": bic(fitter.objective),
        }
    )
    return pd.DataFrame(params).set_index("name")


In [ ]:
fitter = fit.CurveFitter(global_objective)
res = fitter.fit(options={"workers": 8})

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from utils.helpers.plotting_helper import plot_all_results, set_plotting_defaults

set_plotting_defaults()

In [ ]:
for o in fitter.objective.objectives:
    o.plot()

In [ ]:
fig = plot_all_results(
    fitter.objective,
    data,
)
fig.set_size_inches(5, 8)

In [ ]:
# seconds to time
import time

time.strftime("%H:%M:%S", time.gmtime(698))
